# Numerics in CADET
\
This chapter provides an overview of and insights into numerical challenges behind CADET.

## Starting point: The model

Partial differential (algebraic) equations (PD(A)E):\
[<u>Eq.-Generator</u>](https://cadet-equations-74ko8eryoxmsbqspggxrj2.streamlit.app/)

<u>Spatial</u> and <u>Temporal</u> derivatives

Method of lines: Custom spatial discretization that implements the equations and a spatial discretization method plus a time discretization method, where we can often rely on established algorithms and software.

## Time discretization
CADET uses IDAS from the [<u>SUNDIALS software package</u>](https://sundials.readthedocs.io) to integrate the spatially semi-discretized equations in time.
Importantly, IDAS can handle ODAE and parameter sensitivities.

- relTol and absTol IDAS explanation
- stiffness
- Jacobian to solve the linear system within the Newton iteration for the non-linear system




## Spatial discretization
- FV vs DG explanation
- smooth solutions
- general advice on the choice of discretization parameters



- spatial and time integration accuracy should be the same, otherwise we are probably wasting efficiency
- glider grafic for this where we adjust nCol, abstol

## Consistent initialization
We are solving initial value problems (IVP), meaning a PDAE with initial values for the solution variables at $t=0$.
These initial values must be consistent with the equation, i.e. the equation holds when the initial values are inserted.

## Fail compilation
- fail at time section -> play around with consistent init mode
- runs forever -> max newton iterations?
- negative values -> NAN -> simulation abort after max newton iterations




In [1]:
import utility.setting_Col1D_SMA_4comp_LWE_benchmark1 as lwe
from cadet import Cadet


Cadet.cadet_path = r"C:\Users\jmbr\OneDrive\Desktop\CADET_compiled\master5_generalizedUnit_f1a1972\aRELEASE"

model = Cadet()
model.root = lwe.get_model(
        spatial_method_bulk=0,
        spatial_method_particle=0,
        particle_type='GENERAL_RATE_PARTICLE',
        axRefinement=1, # times 8 is number of cells
        parZ=1,
        return_bulk=True)

model.filename = 'test.h5'
model.save()
model.run_simulation()



ReturnInformation(return_code=0, error_message='', log='')

In [5]:
import utility.convergence as convergence
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive
import ipywidgets as widgets
import os

path = os.getcwd()
file_name1D = "/test.h5"
  
column1D = convergence.get_bulk(path+file_name1D, unit="000")
coordinates1D = convergence.get_axial_coordinates(path+file_name1D, unit="000")

if len(column1D.shape) == 3: # only consider first component
    column1D = column1D[:, :, 0]


def graph_column(time=0):
    plt.figure()
    #plt.rcParams["figure.figsize"] = (10,10)
    plt.plot(coordinates1D, column1D[time, :], label='$1D$', linestyle='dotted')
    plt.xlabel('$x~/~M $')
    plt.ylabel('$concentration~/~mol \cdot M^{-3} $')
    plt.legend()
    
interact(graph_column, time = widgets.IntSlider(min=0.0, max=len(column1D[:, 0]) - 3, step=1))

interactive(children=(IntSlider(value=0, description='time', max=1498), Output()), _dom_classes=('widget-inter…

<function __main__.graph_column(time=0)>